In [1]:
import uuid
import logging
from base64 import b64encode
from dotenv import load_dotenv

from evals import Dataset

logging.basicConfig(level=logging.INFO)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
load_dotenv()

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


True

In [38]:
def extract_run_type(path):
    run = path.split("/")[-1].replace("llm-as-judge_", "")
    return "_".join(run.split("_")[:-1])

def organize_runs_by_type(eval_res):
    runs = {}
    for path in eval_res.keys():
        ext_path = extract_run_type(path)
        if ext_path in runs:
            runs[ext_path].append(eval_res[path])
        else:
            runs[ext_path] = [eval_res[path]]
    return runs

def extract_metrics(eval_result):
    correctness = []
    relevancy = []
    completeness = []
    for res in eval_result.results: #iterating trough sessions
        for query in res.get("test_results", []): #iterating trough queries
            if query.get("success"):
                #print(f"{query.get('name')}")
                #print(f"Input: {query.get('input')}")
                #print(f'Expected: {query.get("expected_output")} | Got: {query.get("output")}')
                #print(f"Metrics: {query.get('metrics_data')}")
                for metric in query.get("metrics_data", []):
                    if metric.get("name") =='correctness [GEval]':
                        correctness.append(metric.get("score"))
                    elif metric.get("name") =='relevancy':
                        relevancy.append(metric.get("score"))
                    elif metric.get("name") =='completeness [GEval]':
                        completeness.append(metric.get("score"))

    return {"correctness": correctness, "relevancy": relevancy, "completeness": completeness}

## Evaluation Analysis

In [3]:
ds = Dataset("test")
eval_res = ds.load_evaluation_results()
runs = organize_runs_by_type(eval_res)

In [43]:
run1 = runs.get("google_gemini-2.5-flash_baseline")
run1

[EvalOutput(dataset_name='test', project_id='b3f45644-6222-4593-8955-cf4d8e0b00d5', user_id='53d63d18-cfa1-416e-96e8-770c8f66507b', eval_run_id='2161608a-ae80-4fb7-9f2e-33de91e0316d', llm_model='google_gemini-2.5-flash', agent_type='baseline', created_at='2026-02-27T15:31:23.504609', results=[{'test_results': [{'name': 'Turn 1 in session Prosjekt-initialisering', 'success': True, 'metrics_data': [{'name': 'correctness [GEval]', 'threshold': 0.5, 'success': True, 'score': 0.9018833128651359, 'reason': "The actual output closely aligns with the expected output, accurately summarizing the key events: the purchase and overtakelse dates, the seller's notification of a leak before overtakelse, the recurrence of the leak after overtakelse, the findings of the skaderapporter regarding the membrane puncture, and the technical report in March 2020 highlighting construction issues. The output also correctly identifies the core issue as misrepresentations about the property's condition, particular

In [40]:
correctness = []
relevancy = []
completeness = []
for run in run1:
    metrics = extract_metrics(run)
    correctness.extend(metrics.get("correctness", []))
    relevancy.extend(metrics.get("relevancy", []))
    completeness.extend(metrics.get("completeness", []))

In [44]:
len(correctness)

12

In [42]:
from statsmodels.stats.power import TTestIndPower
import numpy as np

pilot_scores = correctness
observed_sd = np.std(pilot_scores)
observed_mean = np.mean(pilot_scores)

# Anta du vil detektere 10% forbedring (custom vs baseline)
effect_size = 0.10 / observed_sd  # Cohen's d

analysis = TTestIndPower()
n_needed = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.8,
)
print(f"SD: {observed_sd:.3f} → trenger N={n_needed:.0f} per gruppe")

SD: 0.099 → trenger N=16 per gruppe
